# Activity 2.1

postgres=# DROP TABLE IF EXISTS orders; CREATE TABLE orders (  
    id SERIAL PRIMARY KEY,  
    customer_name TEXT,  
    product_category TEXT,  
    quantity INTEGER,  
    price_per_unit NUMERIC(10, 2),  
    order_date DATE,  
    country TEXT  
);  
  SELECT COUNT(*) FROM orders WHERE price_per_unit = 2000.00;

**A** SELECT * FROM orders ORDER BY price_per_unit DESC LIMIT 1;

   id   | customer_name | product_category | quantity | price_per_unit | order_date | country   
--------+---------------+------------------+----------+----------------+------------+---------   
 841292 | Emma Brown    | Automotive       |        3 |        2000.00 | 2024-10-11 | Italy   
(1 row)

**B** SELECT product_category, SUM(quantity) AS total_quantity FROM orders GROUP BY product_category ORDER BY total_quantity DESC LIMIT 3;

 product_category | total_quantity  
------------------+----------------  
 Health & Beauty  |         300842  
 Electronics      |         300804   
 Toys             |         300598  
(3 rows)

**C** SELECT product_category, SUM(price_per_unit*quantity) AS total_revenue FROM orders GROUP BY product_category ORDER BY total_revenue DESC;

 product_category | total_revenue  
------------------+---------------  
 Automotive       |  306589798.86  
 Electronics      |  241525009.45  
 Home & Garden    |   78023780.09  
 Sports           |   61848990.83  
 Health & Beauty  |   46599817.89  
 Office Supplies  |   38276061.64  
 Fashion          |   31566368.22  
 Toys             |   23271039.02  
 Grocery          |   15268355.66  
 Books            |   12731976.04  
(10 rows)

**D** SELECT customer_name, SUM(price_per_unit*quantity) AS total_spent FROM orders GROUP BY customer_name ORDER BY total_spent DESC LIMIT 5;

 customer_name  | total_spent  
----------------+-------------  
 Carol Taylor   |   991179.18  
 Nina Lopez     |   975444.95  
 Daniel Jackson |   959344.48  
 Carol Lewis    |   947708.57  
 Daniel Young   |   946030.14  
(5 rows)

**E** SELECT customer_name, COUNT(*) AS order_count, SUM(price_per_unit * quantity) AS total_spent 
FROM orders 
WHERE customer_name IN ('Carol Taylor', 'Nina Lopez', 'Daniel Jackson', 'Carol Lewis', 'Daniel Young')
GROUP BY customer_name 
ORDER BY total_spent DESC;

 customer_name  | order_count | total_spent
----------------+-------------+-------------
 Carol Taylor   |        1028 |   991179.18  
 Nina Lopez     |         980 |   975444.95  
 Daniel Jackson |        1033 |   959344.48  
 Carol Lewis    |         943 |   947708.57  
 Daniel Young   |         973 |   946030.14  
(5 rows)  
The data looks like this because the generator assigns names and values randomly with equal probability for every row. Because of this random distribution, orders spread out almost evenly, and who are the top customers depends on how many orders a customer got and how many expensive items were in their order.

# Activity 2.2

**Step 1** 50k: joint result - 27501822, time - 2838.106 ms  
100k: joint result - 109946508, time - 16908.093 ms  
200k: joint result - 439395606, time - 77374.726 ms  
When the input doubles, the result grows by a factor of 4, time - by a factor of 4-5. Prediction for 1M: result - 1,1*10 $^{10}$, time - 1934368,15 ms

**Step 2**  
Index doesn`t help, with index it took 16 568 ms, practically the same as without the
index (16908 ms). EXPLAIN ANALYZE shows a Parallel Hash Join: the index is only
used for index-only scans of country, which take just 25-76 ms.   
An index helps to find a few rows quickly. But here the problem is not finding
rows: there are only about 100 countries, so every person matches about 1 000 other
people, and the database still has to build and count all 110M pairs. The
index cannot make the result smaller, so it cannot make the query faster.

**Step 3**  
SELECT SUM(cnt * cnt) AS pairs
FROM (
  SELECT country, COUNT(*) AS cnt
  FROM people_100k
  GROUP BY country
) t;  
Results:  
100k: result -  109946508 (same), time - 231.190 ms;  
people_big: result - 10983941260, time - 4012.365 ms.  
The predicted result is almost the same (1,1*10 $^{10}$), however time decreased by a factor of 482.

**Step 4**  
The join is slow because its result is huge (about 11 billion for 1M people). The rewrite scans the table once and runs in seconds instead of minutes. Therefore, fixing the query is much more effective than adding hardware.  
A bigger machine or a cluster helps only linearly (N nodes give at most N times speedup), but doubling the data makes the work 4 times bigger, so scaling out only delays the problem. Better option is to filter before joining or aggregate.  
Limits of OLTP - PostgreSQL is built for many short transactions, and one heavy query slows down all other users on a single node. In the cloud, scaling one machine up is expensive. Therefore, it is better to separate analytics from OLTP.